In [18]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

REKAP_DIR = Path("../data/raw/rekapitulasi")

BULAN = {
    "JANUARI": 1, "PEBRUARI": 2, "FEBRUARI": 2, "MARET": 3, "APRIL": 4, "MEI": 5,
    "JUNI": 6, "JULI": 7, "AGUSTUS": 8, "SEPTEMBER": 9, "OKTOBER": 10,
    "NOVEMBER": 11, "NOPEMBER": 11, "DESEMBER": 12,
}

csv_files = sorted(REKAP_DIR.glob("*.csv"))
print(f"Jumlah CSV rekapitulasi: {len(csv_files)}")

Jumlah CSV rekapitulasi: 44


In [19]:
# Penyeragaman nama beras
BERAS_STD = {
    "beras premium":          "Beras Premium",
    "beras medium bulog":     "Beras Medium Bulog",
    "beras medium non bulog": "Beras Medium Non Bulog",
}
BERAS_ALIAS = {
    "beras ir.64 kwalitas i":   "beras premium",
    "beras ir.64 kwalitas ii":  "beras medium bulog",
    "beras ir.64 kwalitas iii": "beras medium non bulog",
    "beras ir.64 premium":      "beras premium",
    "beras ir.64 medium bulog": "beras medium bulog",
    "beras ir.64 medium":       "beras medium non bulog",
    "beras medium":             "beras medium non bulog",
}
# Salah ketik di file tertentu (kunci = nama file tanpa .csv)
FILE_ALIAS = {
    "04. REKAP APRIL 2024": {
        "beras ir.64 medium non blog": "beras medium bulog",
    },
}

def standardize_name(x, source_name=""):
    nama = clean_name(x)
    key = nama.casefold()
    key = re.sub(r"\bir\.?\s*64\b", "ir.64", key)
    key = key.replace("kualitas", "kwalitas")
    key = FILE_ALIAS.get(source_name, {}).get(key, key)
    key = BERAS_ALIAS.get(key, key)
    return BERAS_STD.get(key, nama)

In [20]:
# Fungsi Parse dan Bersihkan Data
def _norm(x):
    return re.sub(r"\s+", " ", str(x)).strip().upper()


def clean_name(x):
    """Buang awalan '- ' dan rapikan spasi."""
    return re.sub(r"\s+", " ", re.sub(r"^\s*-\s*", "", str(x))).strip()

def name_key(x):
    """Kunci pencocokan nama komoditas (huruf kecil, spasi rapi)."""
    return clean_name(x).casefold()


def _find_period(text):
    for m in re.finditer(r"([A-Z]+)\s+(\d{4})", _norm(text)):
        if m.group(1) in BULAN:
            return int(m.group(2)), BULAN[m.group(1)]
    return None

def header_period(df):
    """Periode menurut judul di dalam sheet (hanya untuk pembanding)."""
    for v in df.head(10).stack().tolist():
        p = _find_period(v)
        if p:
            return p
    return None

def detect_period(df, fallback_name=""):
    p = _find_period(fallback_name)      # utama: nama file
    if p:
        return p
    p = header_period(df)                # cadangan: judul sheet
    if p:
        return p
    raise ValueError("Periode (bulan/tahun) tidak ditemukan")


def detect_layout(df):
    """Cari baris header, baris tanggal, kolom nama, satuan, HET, dan kolom tanggal 1-31."""
    header_row = name_col = None
    for r in range(min(15, len(df))):
        for c in range(df.shape[1]):
            if "NAMA BAHAN POKOK" in _norm(df.iat[r, c]):
                header_row, name_col = r, c
                break
        if header_row is not None:
            break
    if header_row is None:
        raise ValueError("Header 'NAMA BAHAN POKOK' tidak ditemukan")

    day_row, day_cols = None, {}
    for r in range(header_row, min(header_row + 4, len(df))):
        nums = pd.to_numeric(df.iloc[r], errors="coerce")
        cols = {c: int(v) for c, v in nums.items()
                if pd.notna(v) and float(v).is_integer() and 1 <= v <= 31}
        if len(cols) >= 28:
            day_row, day_cols = r, cols
            break
    if day_row is None:
        raise ValueError("Baris tanggal 1-31 tidak ditemukan")

    unit_col = het_col = None
    for r in range(header_row, day_row + 1):
        for c in range(df.shape[1]):
            t = _norm(df.iat[r, c])
            if t in ("SAT", "SAT.", "SATUAN"):
                unit_col = c
            if t.startswith("HET"):
                het_col = c

    return dict(header_row=header_row, name_col=name_col, day_row=day_row,
                day_cols=day_cols, unit_col=unit_col, het_col=het_col)

def parse_rekap(df, source_name):
    """Satu sheet rekap (wide, mentah) -> long: tanggal, komoditas, satuan_file, harga, HET."""
    year, month = detect_period(df, source_name)
    lay = detect_layout(df)
    day_idx = list(lay["day_cols"].keys())

    records = []
    for r in range(lay["day_row"] + 1, len(df)):
        raw_name = df.iat[r, lay["name_col"]]
        if pd.isna(raw_name) or str(raw_name).strip() == "":
            continue

        prices = pd.to_numeric(df.iloc[r, day_idx], errors="coerce")
        if prices.isna().all():      # baris kelompok, judul, atau catatan kaki
            continue

        nama = standardize_name(raw_name, source_name)   # <-- UBAH: tambah source_name
        satuan = df.iat[r, lay["unit_col"]] if lay["unit_col"] is not None else np.nan
        satuan = str(satuan).strip() if pd.notna(satuan) else np.nan
        het = (pd.to_numeric(df.iat[r, lay["het_col"]], errors="coerce")
               if lay["het_col"] is not None else np.nan)

        for col in day_idx:
            price = prices[col]
            if pd.isna(price):
                continue
            try:
                tgl = pd.Timestamp(year=year, month=month, day=lay["day_cols"][col])
            except ValueError:       # tanggal tidak ada di bulan itu (mis. 30 Februari)
                continue
            records.append((tgl, nama, satuan, price, het))
    out = pd.DataFrame(records, columns=["tanggal", "komoditas", "satuan_file", "harga", "HET"])
    return out, lay, (year, month)

def build_satuan_ref(long_df):
    """Tabel acuan satuan dari file acuan: name_key -> satuan."""
    ref = (long_df.dropna(subset=["satuan_file"])
           .assign(key=lambda d: d["komoditas"].map(name_key))
           .drop_duplicates("key")
           .set_index("key")["satuan_file"])
    return ref.to_dict()

def apply_satuan(long_df, ref, tambahan=None):
    """Satuan HANYA dari acuan (+ tambahan manual). Kunci acuan ikut diseragamkan namanya."""
    lookup = {name_key(standardize_name(k)): v for k, v in ref.items()}
    for nama, sat in (tambahan or {}).items():
        lookup[name_key(standardize_name(nama))] = sat

    out = long_df.copy()
    out["satuan"] = out["komoditas"].map(name_key).map(lookup)
    return out.drop(columns="satuan_file")

In [21]:
# Bangun acuan satuan dari Januari 2023 dan uji
def read_raw(path):
    return pd.read_csv(path, header=None, encoding="utf-8-sig", low_memory=False)


# Cari file acuan (Januari 2023)
ref_matches = [p for p in csv_files if "JANUARI 2023" in p.stem.upper()]
assert len(ref_matches) == 1, f"File acuan tidak unik/ketemu: {ref_matches}"
REF_FILE = ref_matches[0]
print("File acuan:", REF_FILE.name)

ref_long, lay, (y, m) = parse_rekap(read_raw(REF_FILE), REF_FILE.stem)
SATUAN_REF = build_satuan_ref(ref_long)

print(f"Periode          : {y}-{m:02d}")
print(f"Header/tanggal   : baris {lay['header_row']} / {lay['day_row']}")
print(f"Kolom nama/SAT/HET: {lay['name_col']} / {lay['unit_col']} / {lay['het_col']}")
print(f"Hasil long       : {ref_long.shape[0]} baris, {ref_long['komoditas'].nunique()} komoditas")
print(f"Acuan satuan     : {len(SATUAN_REF)} komoditas")

test = apply_satuan(ref_long, SATUAN_REF)
print("Satuan kosong:", test["satuan"].isna().sum())
print("Duplikat tanggal+komoditas:", test.duplicated(["tanggal", "komoditas"]).sum())

test.head(10)

File acuan: 01. REKAP JANUARI 2023.csv
Periode          : 2023-01
Header/tanggal   : baris 3 / 4
Kolom nama/SAT/HET: 1 / 2 / 34
Hasil long       : 1674 baris, 54 komoditas
Acuan satuan     : 54 komoditas
Satuan kosong: 0
Duplikat tanggal+komoditas: 0


,tanggal,komoditas,harga,HET,satuan
0,2023-01-01,Beras Premium,12471.42857,12800.0,Kg
1,2023-01-02,Beras Premium,12471.42857,12800.0,Kg
2,2023-01-03,Beras Premium,12471.42857,12800.0,Kg
3,2023-01-04,Beras Premium,12471.42857,12800.0,Kg
4,2023-01-05,Beras Premium,12471.42857,12800.0,Kg
5,2023-01-06,Beras Premium,12471.42857,12800.0,Kg
6,2023-01-07,Beras Premium,12471.42857,12800.0,Kg
7,2023-01-08,Beras Premium,12471.42857,12800.0,Kg
8,2023-01-09,Beras Premium,12471.42857,12800.0,Kg
9,2023-01-10,Beras Premium,12471.42857,12800.0,Kg


In [22]:
# Uji Parser pada File lain
for i, p in enumerate(csv_files):
    print(i, p.name)

0 01. REKAP JANUARI 2023.csv
1 01. REKAP JANUARI 2024.csv
2 01. REKAPITULASI JANUARI 2025.csv
3 01. REKAPITULASI JANUARI 2026.csv
4 02. REKAP FEBRUARI 2023.csv
5 02. REKAP FEBRUARI 2024.csv
6 02. REKAPITULASI FEBRUARI 2025.csv
7 02. REKAPITULASI FEBRUARI 2026.csv
8 03. REKAP MARET 2023.csv
9 03. REKAP MARET 2024.csv
10 03. REKAPITULASI MARET 2025.csv
11 03. REKAPITULASI MARET 2026.csv
12 04. REKAP APRIL 2023.csv
13 04. REKAP APRIL 2024.csv
14 04. REKAPITULASI APRIL 2025.csv
15 04. REKAPITULASI APRIL 2026.csv
16 05. REKAP MEI 2023.csv
17 05. REKAP MEI 2024.csv
18 05. REKAPITULASI MEI 2025.csv
19 05. REKAPITULASI MEI 2026.csv
20 06. REKAP JUNI 2023.csv
21 06. REKAP JUNI 2024.csv
22 06. REKAPITULASI JUNI 2025.csv
23 06. REKAPITULASI JUNI 2026.csv
24 07. REKAP JULI 2023.csv
25 07. REKAPITULASI JULI 2025.csv
26 07. REKAPITULASI JULI 2026.csv
27 07.REKAP JULI 2024.csv
28 08. REKAP  AGUSTUS 2023.csv
29 08. REKAP AGUSTUS 2024.csv
30 08. REKAPITULASI AGUSTUS 2025.csv
31 08. REKAPITULASI AGUSTUS

In [23]:
def test_file(path):
    df = read_raw(path)
    long_df, lay, (y, m) = parse_rekap(df, path.stem)
    hasil = apply_satuan(long_df, SATUAN_REF)

    names = set(long_df["komoditas"].map(name_key))
    ref_names = set(SATUAN_REF.keys())
    baru = sorted(names - ref_names)        # ada di file ini, tidak ada di acuan
    hilang = sorted(ref_names - names)      # ada di acuan, tidak ada di file ini

    print(f"\n=== {path.name} ===")
    print(f"Periode           : {y}-{m:02d}")
    print(f"Header/tanggal    : baris {lay['header_row']} / {lay['day_row']} | kolom tanggal: {len(lay['day_cols'])}")
    print(f"Kolom nama/SAT/HET: {lay['name_col']} / {lay['unit_col']} / {lay['het_col']}")
    print(f"Hasil long        : {len(hasil)} baris, {long_df['komoditas'].nunique()} komoditas")
    print(f"Tanggal           : {hasil['tanggal'].min().date()} s.d. {hasil['tanggal'].max().date()}")
    print(f"Satuan kosong     : {hasil['satuan'].isna().sum()} baris")
    print(f"Duplikat          : {hasil.duplicated(['tanggal', 'komoditas']).sum()}")
    print(f"Komoditas BARU (tidak ada di acuan)  : {baru}")
    print(f"Komoditas HILANG (ada di acuan saja) : {hilang}")
    return hasil


# Ganti indeks sesuai daftar di atas: satu file 2023 akhir dan satu file 2024
hasil_a = test_file(csv_files[5])
hasil_b = test_file(csv_files[-1])


=== 02. REKAP FEBRUARI 2024.csv ===
Periode           : 2024-02
Header/tanggal    : baris 3 / 4 | kolom tanggal: 29
Kolom nama/SAT/HET: 1 / 2 / 32
Hasil long        : 2349 baris, 81 komoditas
Tanggal           : 2024-02-01 s.d. 2024-02-29
Satuan kosong     : 841 baris
Duplikat          : 0
Komoditas BARU (tidak ada di acuan)  : ['air mineral', 'bahan bakar gas lpg', 'bawang', 'c a b e', 'daging', 'garam beryodium', 'gula', 'i k a n', 'jagung', 'kacang-kacangan', 'mentega', 'mie instan', 'minyak goreng', 'minyak kita', 's u s u', 'sabun cuci', 'sayur-sayuran', 'susu kental manis "bendera" putih', 'tahu', 'tahu biasa', 'tahu susu', 'telur', 'telur ayam broiler', 'tempe', 'tempe kw i (campuran kedelai)', 'tempe kw ii (campuran jagung)', 'tempe kw iii (campuran kulit)', 'tepung terigu', 'ubi-ubian']
Komoditas HILANG (ada di acuan saja) : ['susu bubuk "dancow" full cream', 'telur ayam horn']

=== 12. REKAPITULASI DESEMBER 2025.csv ===
Periode           : 2025-12
Header/tanggal    : baris 2

In [24]:
SATUAN_TAMBAHAN = {}

frames, proc_log = [], []
for p in csv_files:
    try:
        raw = read_raw(p)
        long_df, lay, (y, m) = parse_rekap(raw, p.stem)
        hasil = apply_satuan(long_df, SATUAN_REF, SATUAN_TAMBAHAN)
        hasil["sumber_file"] = p.name
        frames.append(hasil)
        proc_log.append({"file": p.name, "periode": f"{y}-{m:02d}", "baris": len(hasil),
                         "komoditas": hasil["komoditas"].nunique(),
                         "satuan_kosong": int(hasil["satuan"].isna().sum()),
                         "HET_ada": bool(hasil["HET"].notna().any()),
                         "judul_beda": header_period(raw) != (y, m),
                         "status": "OK" if len(hasil) else "KOSONG"})
    except Exception as e:
        proc_log.append({"file": p.name, "periode": None, "baris": None, "komoditas": None,
                         "satuan_kosong": None, "HET_ada": None, "judul_beda": None,
                         "status": f"ERROR: {e}"})

df_log = pd.DataFrame(proc_log)
print(df_log["status"].str.split(":").str[0].value_counts().to_string())
print("\nFile dengan judul sheet berbeda dari nama file:")
print(df_log[df_log["judul_beda"] == True][["file", "periode"]].to_string())
print("\nFile tanpa data:")
print(df_log[df_log["status"] != "OK"][["file", "status"]].to_string())

df_all = pd.concat(frames, ignore_index=True)
df_all["tanggal"] = pd.to_datetime(df_all["tanggal"])
df_all = df_all.sort_values(["tanggal", "komoditas"]).reset_index(drop=True)

COLS = ["tanggal", "komoditas", "satuan", "harga", "HET"]
df_final = df_all[COLS]
print(f"\nTotal baris: {len(df_final):,} | dtype tanggal: {df_all['tanggal'].dtype}")

status
OK    44

File dengan judul sheet berbeda dari nama file:
                                 file  periode
8            03. REKAP MARET 2023.csv  2023-03
14    04. REKAPITULASI APRIL 2025.csv  2025-04
17             05. REKAP MEI 2024.csv  2024-05
25     07. REKAPITULASI JULI 2025.csv  2025-07
31  08. REKAPITULASI AGUSTUS 2026.csv  2026-08
42        12. REKAP DESEMBER 2024.csv  2024-12

File tanpa data:
Empty DataFrame
Columns: [file, status]
Index: []

Total baris: 96,542 | dtype tanggal: datetime64[us]


In [25]:
# ===== FINALISASI: judul, nama, dan satuan =====
df_fin = df_all.copy()

# 1) Hapus judul kelompok (seluruh hurufnya kapital, mis. "MINYAK GORENG", "C A B E")
def is_judul(nama):
    s = str(nama).strip()
    return any(ch.isalpha() for ch in s) and s == s.upper()

judul = df_fin["komoditas"].map(is_judul)
print(f"Judul kelompok dihapus: {judul.sum():,} baris ({df_fin.loc[judul, 'komoditas'].nunique()} nama)")
df_fin = df_fin[~judul].copy()

# 2) Seragamkan penulisan nama
NAMA_BAKU = {
    "minyak kita": "Minyak Kita",
    'susu kental manis "bendera" putih': 'Susu Kental Manis "Bendera" Putih',
    "gula pasir lokal curah": "Gula Pasir Lokal Curah",
}
key = df_fin["komoditas"].map(name_key)
df_fin["komoditas"] = key.map(NAMA_BAKU).fillna(df_fin["komoditas"])

# 3) Satuan manual: HANYA mengisi yang kosong (satuan dari acuan tidak ditimpa)
SATUAN_MANUAL = {
    # buah
    "Apel": "Kg", "Jeruk": "Kg", "Melon": "Kg",
    "Pepaya": "Kg", "Pisang": "Kg", "Semangka": "Kg",
    # tepung
    'Tepung Terigu "Segitiga Biru" Kemasan': "Kg",
    # tahu & tempe
    "Tahu": "Kg", "Tahu biasa": "Kg", "Tahu susu": "Kg",
    "Tempe": "Kg",
    "Tempe Kw I (campuran kedelai)": "Kg",
    "Tempe Kw II (campuran jagung)": "Kg",
    "Tempe Kw III (campuran kulit)": "Kg",
    # konsisten di file lain
    "Telur Ayam Broiler": "Kg",
    'Susu Kental Manis "Bendera" Putih': "390 Gr/kaleng",
}
lookup = {name_key(k): v for k, v in SATUAN_MANUAL.items()}
kosong = df_fin["satuan"].isna()
n_sebelum = int(kosong.sum())
df_fin.loc[kosong, "satuan"] = df_fin.loc[kosong, "komoditas"].map(name_key).map(lookup)

# 4) Satuan yang DIPAKSA (menimpa apa pun yang tercatat di file)
SATUAN_PAKSA = {"Minyak Kita": "Liter"}
for nama, sat in SATUAN_PAKSA.items():
    m = df_fin["komoditas"] == nama
    print(f"{nama}: satuan sebelum = {df_fin.loc[m, 'satuan'].value_counts(dropna=False).to_dict()}")
    df_fin.loc[m, "satuan"] = sat

# 5) Hasil akhir
df_all = df_fin.sort_values(["tanggal", "komoditas"]).reset_index(drop=True)
df_final = df_all[["tanggal", "komoditas", "satuan", "harga"]]

print(f"\nSatuan kosong: {n_sebelum:,} -> {df_final['satuan'].isna().sum():,}")
print(f"Total baris   : {len(df_final):,} | komoditas: {df_final['komoditas'].nunique()}")
print(f"Duplikat      : {df_final.duplicated(['tanggal', 'komoditas']).sum()}")

print("\nMinyak Kita:")
print(df_final[df_final["komoditas"] == "Minyak Kita"]["satuan"].value_counts().to_string())

print("\nDaftar satuan:")
print(df_final["satuan"].value_counts().to_string())

ganda = df_final.groupby("komoditas")["satuan"].nunique()
print("\nKomoditas dengan >1 satuan:", ganda[ganda > 1].to_dict() or "tidak ada")

Judul kelompok dihapus: 11,066 baris (23 nama)
Minyak Kita: satuan sebelum = {nan: 1248}

Satuan kosong: 13,988 -> 0
Total baris   : 85,476 | komoditas: 71
Duplikat      : 0

Minyak Kita:
satuan
Liter    1248

Daftar satuan:
satuan
Kg               68219
Ikat Kecil        4005
390 Gr/kaleng     1487
250 Gr/Bngks      1335
86 Gr/Bngks       1335
Butir             1334
19 Ltr/Galon      1333
Per Tabung        1333
350 Gram          1332
900 Gram          1332
Liter             1248
400 Gr/Dos        1183

Komoditas dengan >1 satuan: tidak ada


In [26]:
# ===== PEMBERSIHAN AKHIR =====
# 1) Koreksi penulisan satuan
KOREKSI_SATUAN = {"400 Gr/Dos": "400 Gr/Dus"}
for lama, baru in KOREKSI_SATUAN.items():
    m = df_all["satuan"] == lama
    print(f"Satuan {lama} -> {baru}: {m.sum():,} baris")
    df_all.loc[m, "satuan"] = baru

# 2) Seragamkan penulisan nama yang sama tapi beda huruf besar-kecil
#    (dipakai ejaan yang paling sering muncul, mis. "Gula pasir lokal curah" -> "Gula Pasir Lokal Curah")
key = df_all["komoditas"].map(name_key)
ejaan = (df_all.assign(key=key)
         .groupby("key")["komoditas"]
         .agg(lambda s: s.value_counts().index[0]))
ganda = df_all.assign(key=key).groupby("key")["komoditas"].nunique()
for k in ganda[ganda > 1].index:
    print(f"Nama diseragamkan: {sorted(df_all.loc[key == k, 'komoditas'].unique())} -> {ejaan[k]}")
df_all["komoditas"] = key.map(ejaan)

# 3) Harga 0/kosong = tidak tercatat -> buang barisnya
tak_tercatat = df_all["harga"].isna() | (df_all["harga"] == 0)
print(f"\nHarga 0/kosong dibuang: {tak_tercatat.sum():,} baris")
print(df_all[tak_tercatat].groupby(df_all["tanggal"].dt.strftime("%Y-%m")).size().to_string())
df_all = df_all[~tak_tercatat].reset_index(drop=True)

# 4) Hasil akhir (tanpa HET)
df_final = (df_all[["tanggal", "komoditas", "satuan", "harga"]]
            .sort_values(["tanggal", "komoditas"]).reset_index(drop=True))

print(f"\nTotal baris    : {len(df_final):,}")
print(f"Rentang        : {df_final['tanggal'].min().date()} s.d. {df_final['tanggal'].max().date()}")
print(f"Komoditas      : {df_final['komoditas'].nunique()}")
print(f"Harga <= 0     : {(df_final['harga'] <= 0).sum()}")
print(f"Satuan kosong  : {df_final['satuan'].isna().sum()}")
print(f"Duplikat       : {df_final.duplicated(['tanggal', 'komoditas']).sum()}")
print("\n", df_final.dtypes.to_string())

Satuan 400 Gr/Dos -> 400 Gr/Dus: 1,183 baris

Harga 0/kosong dibuang: 140 baris
tanggal
2023-09     1
2023-10    28
2023-11    30
2023-12    31
2024-01    27
2024-02    22
2024-03     1

Total baris    : 85,336
Rentang        : 2023-01-01 s.d. 2026-08-31
Komoditas      : 71
Harga <= 0     : 0
Satuan kosong  : 0
Duplikat       : 0

 tanggal      datetime64[us]
komoditas               str
satuan                  str
harga               float64


In [27]:
OUT_DIR = Path("../data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)
df_final.to_csv(OUT_DIR / "harga_komoditas.csv", index=False, encoding="utf-8-sig")
print(f"\nTersimpan: {OUT_DIR / 'harga_komoditas.csv'}")


Tersimpan: ..\data\processed\harga_komoditas.csv
